[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snuuq/ztm-tutorial/blob/main/week1/assignment.ipynb)

> **Colab 사용 시**: 열자마자 `파일 → Drive에 사본 저장`으로 사본을 만들어 작업하세요.
> 노트북이 만드는 파일(`pred_*.png`, `models/`)은 좌측 파일 패널에서 다운로드할 수 있고, 세션이 초기화되면 사라집니다.

# 1주차 과제 — 선형회귀 전 과정 직접 구현

1장에서 본 워크플로 6단계를 **다른 데이터로 처음부터 직접** 구현합니다.
교재 [Exercises](https://www.learnpytorch.io/01_pytorch_workflow/#exercises)를 기반으로 한 과제입니다.

## 해야 할 일
1. `w=0.7, b=0.3`으로 직선 데이터 생성, `torch.randperm`으로 랜덤 80:20 분할
2. `nn.Module` 상속 선형회귀 모델 작성
3. SGD로 300 epoch 학습, loss 곡선 그리기
4. 학습된 파라미터가 실제 `w, b`에 얼마나 가까운지 비교
5. 예측 vs 실제 산점도 (학습 전/후 2장 저장)
6. `state_dict`로 저장 → 로드 → 동일 예측 확인

## 규칙
- `# TODO` 부분을 채우세요. 그 외 제공 코드는 수정하지 않아도 됩니다.
- 재현성을 위해 시드는 제공된 값(2525)을 그대로 쓰세요.

## 0. 준비

Library import와 device 설정, 시드 고정.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, "| device:", device)

RANDOM_SEED = 2525

## 1. 데이터 생성 & 랜덤 분할

`y = 0.7 * x + 0.3` 직선 데이터를 만들고 80:20으로 나눕니다.

교재(01장)는 앞 80%를 그대로 잘라 train으로 썼지만, 실전 데이터는 저장된 순서에 편향이 있을 수 있어(값 크기순 정렬, 수집 시점순 등) 보통 **인덱스를 랜덤하게 섞어서** 분할합니다. 여기서는 `torch.randperm`으로 직접 랜덤 분할을 구현합니다.

> 이 데이터에서 앞/뒤로 자르는 순차 분할을 쓰면 test는 $x \in [0.8, 1.0)$, 즉 학습 범위 **밖**(외삽, extrapolation)만 평가하게 됩니다. 랜덤 분할의 test는 무엇을 평가하게 될까요?

In [ ]:
weight = 0.7
bias = 0.3

# TODO: X를 0부터 1까지 0.01 간격으로 만들고, 모델 입력 형태 (100, 1)이 되도록 하세요
X = None
# TODO: y = weight * X + bias
y = None

# 랜덤 분할 — 재현성을 위해 셔플 직전에 시드를 고정합니다 (이 줄은 그대로 두세요)
torch.manual_seed(RANDOM_SEED)

# TODO: torch.randperm(len(X))으로 0~99를 뒤섞은 인덱스 텐서를 만들고, 앞 80개 인덱스를 train, 나머지 20개를 test로 나누세요
#       (힌트: index tensor로 바로 뽑을 수 있습니다. 예: X[train_idx])
indices = None
train_split = None
train_idx, test_idx = None, None
X_train, y_train = None, None
X_test, y_test = None, None

# 확인: 80 20
# print(len(X_train), len(X_test))
# 확인: 출력 결과가 오름차순, 내림차순 등 순차적인 출력만 아니면 됩니다.
# print(X_test)

## 2. 데이터 시각화 (+ 학습 전 그래프는 5단계에서 저장)

시각화 함수는 제공합니다. 이후 예측 확인에 계속 사용합니다.

In [ ]:
def plot_predictions(train_data=None, train_labels=None,
                     test_data=None, test_labels=None,
                     predictions=None, save_path=None):
    """학습/테스트 데이터와 (있다면) 예측값을 그립니다. save_path를 주면 파일로 저장합니다."""
    if train_data is None:
        train_data, train_labels = X_train, y_train
        test_data, test_labels = X_test, y_test
    plt.figure(figsize=(10, 7))
    plt.scatter(train_data, train_labels, c="b", s=4, label="Train data")
    plt.scatter(test_data, test_labels, c="g", s=4, label="Test data")
    if predictions is not None:
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")
    plt.legend(prop={"size": 14})
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"그래프 저장: {save_path}")

plot_predictions()

## 3. 모델 정의

`nn.Module`을 상속해 선형회귀 모델을 만드세요. `nn.Parameter` 방식과 `nn.Linear` 방식 중 하나를 선택하면 됩니다.

In [ ]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: 학습 가능한 파라미터(또는 nn.Linear 층)를 정의하세요

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: y = weights * x + bias 계산을 작성하세요
        return None


torch.manual_seed(RANDOM_SEED)
model = LinearRegressionModel()
print(model.state_dict())   # 랜덤 초기값 확인

## 4. Loss function & Optimizer

- loss: `nn.L1Loss` (MAE)
- optimizer: `torch.optim.SGD`, `lr=0.05`

In [ ]:
# TODO: loss_fn과 optimizer를 만드세요
loss_fn = None
optimizer = None

## 5. 학습 전 예측 저장

랜덤 파라미터 상태의 예측을 `pred_before.png`로 저장합니다. 예측(빨강)이 정답(초록)에서 멀리 떨어져 있어도 정상입니다.

In [ ]:
model.eval()
# TODO: inference_mode 안에서 X_test에 대한 예측을 구하세요
with torch.inference_mode():
    preds_before = None

plot_predictions(predictions=preds_before, save_path="pred_before.png")

## 6. 학습 루프 — 300 epochs

학습 5단계(`forward → loss → zero_grad → backward → step`)를 채우고,
20 epoch마다 train/test loss를 기록·출력하세요.

In [ ]:
torch.manual_seed(RANDOM_SEED)
epochs = 300

epoch_count, train_loss_values, test_loss_values = [], [], []

for epoch in range(epochs):
    ### 학습 ###
    model.train()

    # TODO: 학습 5단계
    # 1. forward pass

    # 2. loss 계산

    # 3. gradient 초기화

    # 4. backpropagation

    # 5. 파라미터 업데이트


    ### 평가 ###
    model.eval()
    with torch.inference_mode():
        # TODO: test 예측과 test loss를 계산하세요
        test_pred = None
        test_loss = None

    if epoch % 20 == 0:
        # TODO: 20 epoch마다 중간 학습 결과를 저장하고 출력하세요
        epoch_count.append(None)
        train_loss_values.append(None)
        test_loss_values.append(None)
        print(f"Epoch: {None} | Train loss: {None} | Test loss: {None}")

## 7. Loss 곡선

In [ ]:
# TODO: epoch_count에 대해 train_loss_values와 test_loss_values를 그리세요
#       (제목, x/y 라벨, 범례 포함)

## 8. 파라미터 비교

학습된 파라미터가 실제 `weight=0.7, bias=0.3`에 얼마나 가까운지 출력하세요.

In [ ]:
# TODO: model.state_dict()의 값과 정답(weight, bias)을 나란히 출력하고, 각 파라미터의 차이(절댓값)도 계산해 보세요.

## 9. 학습 후 예측 저장

학습 후 예측을 `pred_after.png`로 저장합니다. 예측(빨강)이 정답(초록)에 겹치면 성공입니다.

In [ ]:
model.eval()
# TODO: 학습된 모델로 X_test 예측 → plot_predictions로 그려서 pred_after.png로 저장하세요
with torch.inference_mode():
    preds_after = None



## 10. 저장 & 로드 & 검증

`state_dict`를 저장하고, 새 인스턴스에 로드한 뒤 **예측이 완전히 같은지** 확인하세요.

In [ ]:
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = MODEL_PATH / "week1_assignment_model.pth"

# TODO: 1) model의 state_dict를 MODEL_SAVE_PATH에 저장
# TODO: 2) 새 LinearRegressionModel 인스턴스를 만들어 state_dict 로드
loaded_model = None

# TODO: 3) 로드한 모델의 예측이 preds_after와 같은지 확인 (또는 torch.isclose)


## 11. (선택 도전) `nn.Linear` 버전

`nn.Linear(1, 1)`로 같은 모델을 다시 만들어 학습해 보고, 학습된 파라미터가 위와 비슷하게 나오는지 확인해 보세요.
그다음 `lr`을 0.1로 키워 다시 학습해 보세요. 더 빨리 수렴하나요, 아니면 loss가 어느 지점에서 더 안 내려가나요? 관찰한 것을 정리해보세요.

In [ ]:
# (선택) 자유롭게 작성

---

## 체크리스트

- [ ] 모든 셀이 위에서 아래로 에러 없이 실행됨 (로컬: `Kernel → Restart & Run All`, Colab: `런타임 → 세션 다시 시작 및 모두 실행`)
- [ ] `pred_before.png`, `pred_after.png` 2장이 생성됨
- [ ] loss 곡선에서 train/test loss가 함께 감소함
- [ ] 학습된 weight, bias가 0.7, 0.3에 근접함 (차이 0.05 이내 권장)
- [ ] 저장→로드 후 예측 동일 검증이 `True`